In [72]:
import pandas as pd
import re
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import statsmodels.api as sm


In [5]:
df = pd.read_parquet(r"C:\Users\Mateus Monteleone\Projects\ic\data\tweets_192k_labeled.parquet")


In [6]:
df.head()

,full_text,clean_text,unsupervised_sentiment
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0


In [ ]:
# Combine the text columns into one searchable field
df["search_text"] = (
    df["full_text"].fillna("").astype(str) + " " +
    df["clean_text"].fillna("").astype(str)
).str.lower()

## Foi apagado o resto disso.


In [16]:
# Build a single searchable text field
df_model = df.copy()
df_model["search_text"] = (
    df_model["full_text"].fillna("").astype(str) + " " +
    df_model["clean_text"].fillna("").astype(str)
).str.lower()

In [37]:
# Candidate mention patterns
candidate_patterns = {
    "lula": r"\blula\b|\blule\b|\blulinha\b|\blulão\b|\bpt\b|\b9 dedos\b|\bnove dedos\b",

    "bolsonaro": r"\bbolsonaro\b|\bbolsonario\b|\bbolsonaru\b|\bbozo\b|\bbozonaro\b|\bbolsolixo\b|\bbolsominion\b|\bjair\b|\bbiroliro\b|\bmito\b",

    "tebet": r"\btebet\b|\bsimone tebet\b|\bsimone\b",

    "ciro_gomes": r"\bciro\b|\bciro gomes\b|\bcirogomes\b|\bciro2022\b",

    "soraya_thronicke": r"\bsoraya\b|\bsoraya thronicke\b|\bsoraya2022\b",

    "luiz_felipe_davila": r"\bd['’]avila\b|\bdavila\b|\bluiz felipe d['’]avila\b|\bluiz felipe davila\b",

    "padre_kelmon": r"\bkelmon\b|\bpadre kelmon\b|\bkelmon souza\b|\bpadrekelmon\b",

    "leonardo_pericles": r"\bleonardo pericles\b|\bleonardo péricles\b|\bp[eé]ricles\b",

    "sofia_manzano": r"\bsofia manzano\b|\bmanzano\b",

    "vera_lucia_salgado": r"\bvera l[uú]cia\b|\bvera lucia\b|\bvera l[uú]cia salgado\b|\bveralucia\b",

    "jose_maria_eymael": r"\beymael\b|\bjose maria eymael\b|\bjos[eé] maria eymael\b|\bey ey eymael\b",
}


In [38]:
# Create one binary mention column per candidate
for candidate, pattern in candidate_patterns.items():
    df_model[f"mentions_{candidate}"] = (
        df_model["search_text"].str.contains(pattern, regex=True, na=False).astype(int)
    )

In [39]:
mention_cols = [f"mentions_{candidate}" for candidate in candidate_patterns.keys()]
df_candidates = df_model[df_model[mention_cols].sum(axis=1) > 0].copy()


In [40]:
df_candidates["mentioned_candidates"] = df_candidates[mention_cols].apply(
    lambda row: [
        col.replace("mentions_", "")
        for col, value in row.items()
        if value == 1
    ],
    axis=1
)

In [41]:
df_candidates = df_candidates[
    ["full_text", "clean_text", "unsupervised_sentiment", "mentioned_candidates"] + mention_cols
].copy()

In [42]:
df_candidates.head(50)

,full_text,clean_text,unsupervised_sentiment,mentioned_candidates,mentions_lula,mentions_bolsonaro,mentions_tebet,mentions_ciro_gomes,mentions_soraya_thronicke,mentions_luiz_felipe_davila,mentions_padre_kelmon,mentions_leonardo_pericles,mentions_sofia_manzano,mentions_vera_lucia_salgado,mentions_jose_maria_eymael
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0,[lula],1,0,0,0,0,0,0,0,0,0,0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0,[lula],1,0,0,0,0,0,0,0,0,0,0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0,[lula],1,0,0,0,0,0,0,0,0,0,0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0,[lula],1,0,0,0,0,0,0,0,0,0,0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0,[lula],1,0,0,0,0,0,0,0,0,0,0
5,LULA PRESIDENTE AMANHÃ 1️⃣3️⃣🚩❤ https://t.co/T...,LULA PRESIDENTE AMANHÃ 1️⃣3️⃣ :bandeira_triang...,1.0,[lula],1,0,0,0,0,0,0,0,0,0,0
6,@DiRamaciotti Bora de Ciro está em terceiro lu...,@USER Bora de Ciro está em terceiro lugar e se...,1.0,[ciro_gomes],0,0,0,1,0,0,0,0,0,0,0
7,@denisoon_ Meus primos que nunca trabalharam n...,@USER Meus primos que nunca trabalharam na vid...,-1.0,[bolsonaro],0,1,0,0,0,0,0,0,0,0,0
8,Se o Lula realmente ganhar no primeiro turno e...,Se o Lula realmente ganhar no primeiro turno e...,-1.0,[lula],1,0,0,0,0,0,0,0,0,0,0
9,@GuilhermeBoulos Lula 13❤️❤️❤️,@USER Lula 13 :coração_vermelho: :coração_verm...,1.0,[lula],1,0,0,0,0,0,0,0,0,0,0


In [43]:

mention_cols = [
    "mentions_lula",
    "mentions_bolsonaro",
    "mentions_tebet",
    "mentions_ciro_gomes",
    "mentions_soraya_thronicke",
    "mentions_luiz_felipe_davila",
    "mentions_padre_kelmon",
    "mentions_leonardo_pericles",
    "mentions_sofia_manzano",
    "mentions_vera_lucia_salgado",
    "mentions_jose_maria_eymael",
]

In [44]:
candidate_mention_counts = (
    df_candidates[mention_cols]
    .sum()
    .sort_values(ascending=False)
    .rename("mention_count")
    .reset_index()
)

In [45]:
candidate_mention_counts["candidate"] = candidate_mention_counts["index"].str.replace("mentions_", "", regex=False)
candidate_mention_counts = candidate_mention_counts[["candidate", "mention_count"]]



In [46]:
sentiment_by_candidate = {}

In [47]:
for col in mention_cols:
    candidate_name = col.replace("mentions_", "")
    temp = (
        df_candidates[df_candidates[col] == 1]["unsupervised_sentiment"]
        .value_counts(dropna=False)
        .rename_axis("unsupervised_sentiment")
        .reset_index(name="count")
    )
    temp["candidate"] = candidate_name
    sentiment_by_candidate[candidate_name] = temp[["candidate", "unsupervised_sentiment", "count"]]


In [48]:
sentiment_candidate_table = pd.concat(sentiment_by_candidate.values(), ignore_index=True)

In [49]:
sentiment_candidate_pivot = (
    sentiment_candidate_table
    .pivot(index="candidate", columns="unsupervised_sentiment", values="count")
    .fillna(0)
    .astype(int)
)

In [50]:
sentiment_candidate_pivot["total_mentions"] = sentiment_candidate_pivot.sum(axis=1)
sentiment_candidate_pivot = sentiment_candidate_pivot.sort_values("total_mentions", ascending=False)

In [51]:
print(candidate_mention_counts)
print(sentiment_candidate_pivot)


             candidate  mention_count
0                 lula         118974
1            bolsonaro          75646
2           ciro_gomes          19898
3                tebet           4139
4     soraya_thronicke            484
5         padre_kelmon            454
6    leonardo_pericles            144
7   luiz_felipe_davila             95
8        sofia_manzano             69
9    jose_maria_eymael             27
10  vera_lucia_salgado              7
unsupervised_sentiment   -1.0    0.0    1.0  total_mentions
candidate                                                  
lula                    37412  34972  46590          118974
bolsonaro               34439  21051  20156           75646
ciro_gomes               8526   3342   8030           19898
tebet                    1527   1383   1229            4139
soraya_thronicke          247    179     58             484
padre_kelmon              161    199     94             454
leonardo_pericles          78     54     12             144
luiz

In [58]:
X = df_candidates[mention_cols].copy()
y = df_candidates["unsupervised_sentiment"].copy()


In [59]:
# Check exact sentiment labels before running
print("Unique sentiment labels:", y.unique())

Unique sentiment labels: [ 1.  0. -1.]


In [63]:
# Create binary targets for each class
y_positive = (y == 1).astype(int)
y_neutral = (y == 0).astype(int)
y_negative = (y == -1).astype(int)

In [64]:
# Train/test split
X_train, X_test, y_pos_train, y_pos_test = train_test_split(
    X, y_positive, test_size=0.2, random_state=42, stratify=y_positive
)

_, _, y_neu_train, y_neu_test = train_test_split(
    X, y_neutral, test_size=0.2, random_state=42, stratify=y_neutral
)

_, _, y_neg_train, y_neg_test = train_test_split(
    X, y_negative, test_size=0.2, random_state=42, stratify=y_negative
)


In [66]:
# Positive model
model_positive = LogisticRegression(max_iter=1000, class_weight= 'balanced')
model_positive.fit(X_train, y_pos_train)
y_pos_pred = model_positive.predict(X_test)

print("\nPOSITIVE MODEL")
print(confusion_matrix(y_pos_test, y_pos_pred))
print(classification_report(y_pos_test, y_pos_pred))


POSITIVE MODEL
[[11289 12985]
 [ 4118  9682]]
              precision    recall  f1-score   support

           0       0.73      0.47      0.57     24274
           1       0.43      0.70      0.53     13800

    accuracy                           0.55     38074
   macro avg       0.58      0.58      0.55     38074
weighted avg       0.62      0.55      0.56     38074



In [67]:
# Neutral model
model_neutral = LogisticRegression(max_iter=1000, class_weight='balanced')
model_neutral.fit(X_train, y_neu_train)
y_neu_pred = model_neutral.predict(X_test)

print("\nNEUTRAL MODEL")
print(confusion_matrix(y_neu_test, y_neu_pred))
print(classification_report(y_neu_test, y_neu_pred))


NEUTRAL MODEL
[[17194 10285]
 [ 6553  4042]]
              precision    recall  f1-score   support

           0       0.72      0.63      0.67     27479
           1       0.28      0.38      0.32     10595

    accuracy                           0.56     38074
   macro avg       0.50      0.50      0.50     38074
weighted avg       0.60      0.56      0.57     38074



In [69]:
# Negative model
model_negative = LogisticRegression(max_iter=1000, class_weight='balanced')
model_negative.fit(X_train, y_neg_train)
y_neg_pred = model_negative.predict(X_test)

print("\nNEGATIVE MODEL")
print(confusion_matrix(y_neg_test, y_neg_pred))
print(classification_report(y_neg_test, y_neg_pred))


NEGATIVE MODEL
[[14690  9704]
 [ 8188  5492]]
              precision    recall  f1-score   support

           0       0.64      0.60      0.62     24394
           1       0.36      0.40      0.38     13680

    accuracy                           0.53     38074
   macro avg       0.50      0.50      0.50     38074
weighted avg       0.54      0.53      0.53     38074



In [ ]:
# Coefficients for interpretation
#
coef_positive = pd.DataFrame({
    "variable": mention_cols,
    "coef": model_positive.coef_[0]
}).sort_values("coef", ascending=False)

coef_neutral = pd.DataFrame({
    "variable": mention_cols,
    "coef": model_neutral.coef_[0]
}).sort_values("coef", ascending=False)

coef_negative = pd.DataFrame({
    "variable": mention_cols,
    "coef": model_negative.coef_[0]
}).sort_values("coef", ascending=False)

print("\nPOSITIVE COEFFICIENTS")
print(coef_positive)

print("\nNEUTRAL COEFFICIENTS")
print(coef_neutral)

print("\nNEGATIVE COEFFICIENTS")
print(coef_negative)


POSITIVE COEFFICIENTS
                       variable      coef
9   mentions_vera_lucia_salgado -0.017480
3           mentions_ciro_gomes -0.045829
10   mentions_jose_maria_eymael -0.159295
0                 mentions_lula -0.250090
2                mentions_tebet -0.316509
6         mentions_padre_kelmon -0.414844
5   mentions_luiz_felipe_davila -0.568149
1            mentions_bolsonaro -0.885297
8        mentions_sofia_manzano -0.922214
4     mentions_soraya_thronicke -1.195858
7    mentions_leonardo_pericles -1.673986

NEUTRAL COEFFICIENTS
                       variable      coef
1            mentions_bolsonaro  0.004572
0                 mentions_lula  0.000968
9   mentions_vera_lucia_salgado -0.000236
6         mentions_padre_kelmon -0.001275
8        mentions_sofia_manzano -0.001723
10   mentions_jose_maria_eymael -0.001884
7    mentions_leonardo_pericles -0.002905
4     mentions_soraya_thronicke -0.004284
5   mentions_luiz_felipe_davila -0.005073
2                mentions_tebet

In [73]:
X = df_candidates[mention_cols].copy()
X = sm.add_constant(X)

y = df_candidates["unsupervised_sentiment"].copy()
y_positive = (y == 1).astype(int)

logit_positive = sm.Logit(y_positive, X)
result_positive = logit_positive.fit()

print(result_positive.summary())

Optimization terminated successfully.
         Current function value: 0.639703
         Iterations 6
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190367
Model:                              Logit   Df Residuals:                   190355
Method:                               MLE   Df Model:                           11
Date:                    Sun, 15 Mar 2026   Pseudo R-squ.:                 0.02308
Time:                            20:18:00   Log-Likelihood:            -1.2178e+05
converged:                           True   LL-Null:                   -1.2465e+05
Covariance Type:                nonrobust   LLR p-value:                     0.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                          -0.0438    